# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [2]:
import sys
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]
import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType
)

print(sys.executable)


import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    # Mémoire réduite : machine à 8 Go de RAM, éviter de saturer le système
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.python.worker.memory", "1g") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false")
    # Active un vrai traceback Python si l'UDF crash, au lieu d'une erreur Java opaque
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

c:\Users\Julien ANTOGNELLI\AppData\Local\Programs\Python\Python311\python.exe
Spark version : 4.1.1


## 1 - Chemins & constantes

In [3]:
TRAIN_PATH   = "./data/Train_5/"
TEST_PATH    = "./data/Test_5/"
OUTPUT_PREDS = "./output/predictions/"
MODEL_PATH   = "./output/model/"
TARGET_SIZE  = (64, 64)

In [4]:
print(os.environ.get("HADOOP_HOME"))
spark.read.format("binaryFile").load(TRAIN_PATH).limit(1).show()

C:\hadoop
+----+----------------+------+-------+
|path|modificationTime|length|content|
+----+----------------+------+-------+
+----+----------------+------+-------+



## 2 - Parsing

Ce bloc lit des images depuis un dossier et pour chaque image:
- Charge le contenu binaire du fichier via Spark
- Applique une fonction Python grâce à une UDF 
    - ouvre l'image avec PIL 
    - redimensionne en 64x64
    - transforme les pixels RGB en une liste de float 
- Extrait le nom du fichier et du dossier parent depuis le chemin du fichier
- Renvoie un dataframe spark avec 3 colonnes

In [ ]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    # Fonction exécutée sur les workers
    # reçoit les octets bruts d'une image
    try:
        # import à l'intérieur pour que la fonction soit auto-dépendante
        from PIL import Image
        # décode les octets en images, force RGB (3 canaux)
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        # redimensionne en widthxheight (Lanczos = qualité de rééchantillonage)
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        # récupère les octets bruts des pixels 
        raw = img.tobytes()
        n = len(raw)    # 64*64*3 = 12288 octets
        # convertit octets en liste d'entiers (0-255) représentant les pixels
        pixels = list(struct.unpack(f"{n}B", raw))
        # renvoie un tuple (width, height, channels, pixels) pour Spark
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

# Schéma Spark décrivant le format du tupe renvoyé par decode_image_bytes
# Il faut dire explicitement à Spark le type de retour d'une fonction Python
_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])

# Transforme la fonction Python en UDF Spark, avec le schéma de retour, utilisable dans une colonne Spark
decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        # lit le contenu des fichiers comme des octets bruts 
        spark.read.format("binaryFile")
        # va chercher les fichiers même dans les sous dossiers 
        .option("recursiveFileLookup", "true")
        # ne garde que les images (filtrage par extension)
        .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
        .load(path)
    )
    return (
        raw
        .select(
            # capture ce qui suit le dernier "/" --> nom du fichier
            F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias("image_id"),
            # capture le nom du dossier parent --> label
            F.regexp_extract(F.col("path"), r"/([^/]+)/[^/]+$", 1).alias("label"),
            # "content" est la colonne où binaryFile met les octets bruts de l'image
            # nom donné par défaut par binaryFile, on la renomme en raw_bytes
            F.col("content").alias("raw_bytes"),
        )
        # applique l'UDF sur chaque ligne, résultat --> colonne "struct" (width, height, channels, pixels)
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        # enlève les lignes où l'image est illisible
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            # on ne garde que les pixels décodés, pas les octets bruts
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")

c:\Users\Julien ANTOGNELLI\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Images train : 10
Images test  : 10


## 2.2 - Parsing: normalisation couleur

In [ ]:
def normalize_rgb(pixels):
    # Reçoit "pixels": liste de float représentant les valeurs RGB (0-255) 
    if pixels is None:
        # Sécurité : si l'image est illisible, on renvoie None
        return None
    # division de chaque pixel par 255.0 pour normaliser entre 0 et 1
    # appliqué élément par élément
    return [p / 255.0 for p in pixels]

# convertit la fonction Python en UDF Spark 
normalize_rgb_udf = F.udf(normalize_rgb, ArrayType(FloatType()))

def preprocess_rgb(df):
    # Ajoute une colonne "pixels_norm" avec les pixels normalisés entre 0 et 1
    return df.withColumn("pixels_norm", normalize_rgb_udf(F.col("pixels")))

# applique le preprocessing sur les DataFrames train et test
train_rgb_normalized_df  = preprocess_rgb(train_parsed_df)
test_rgb_normalized_df   = preprocess_rgb(test_parsed_df)

print("Aperçu après normalisation RGB :")
train_rgb_normalized_df.select("image_id", "label", "pixels_norm").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64*3 = 12288 valeurs RGB normalisées
expected_len = TARGET_W * TARGET_H * 3
check_len = (
    train_rgb_normalized_df
    # F.size() calcule la taille de la liste dans la colonne pixels_norm
    .select(F.size(F.col("pixels_norm")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_norm (attendu {expected_len}) : {check_len}")

Aperçu après normalisation RGB :


Py4JJavaError: An error occurred while calling o498.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 31.0 failed 1 times, most recent failure: Lost task 0.0 in stage 31.0 (TID 67) (DESKTOP-PKST5ET executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:685)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:128)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:106)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:596)
	at org.apache.spark.InterruptibleIterator.hasNex

## 3 - Prétraitement 

On garde `pixels` (RGB brut, 0-255) intact pour l'affichage futur dans Streamlit,
et on ajoute une colonne `pixels_gray` : niveaux de gris normalisés en [0.0, 1.0].

Conversion RGB -> nuances de gris : formule de luminance pondérée (standard) :
```
gray = 0.299*R + 0.587*G + 0.114*B
```

`pixels` est une liste aplatie `[R,G,B, R,G,B, ...]` de taille 64*64*3 = 12288.
L'UDF regroupe les valeurs par 3, applique la formule et renvoie les pixels en niveaux de gris

In [ ]:
def rgb_to_grayscale(pixels):
    # Reçoit pixels: liste de float représentant les valeurs RGB normalisées (0-1)
    if pixels is None:
        return None
    gray = []
    # On parcourt les pixels par groupes de 3 (R, G, B)
    for i in range(0, len(pixels), 3):
        # découpe le paquet en 3 variables nommées r, g, b
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        # Combine les 3 canaux en une seule valeur de gris, pondérée selon la sensibilité
        # de l'œil humain (plus sensible au vert qu'au rouge, peu sensible au bleu)
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value)
    # Renvoie une liste 3x plus courte que l'entrée : un seul niveau de gris par pixel 
    return gray

gray_udf = F.udf(rgb_to_grayscale, ArrayType(FloatType()))

def preprocess_grayscale(df):
    # ajoute une colonne pixels_grayscale avec les pixels convertis en niveaux de gris
    return df.withColumn("pixels_grayscale", gray_udf(F.col("pixels_norm")))

train_preprocessed_gray_df = preprocess_grayscale(train_rgb_normalized_df)
test_preprocessed_gray_df  = preprocess_grayscale(test_rgb_normalized_df)

print("Aperçu après prétraitement :")
train_preprocessed_gray_df.select("image_id", "label", "pixels_grayscale").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_gray_df
    .select(F.size(F.col("pixels_grayscale")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_grayscale (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                        pixels_grayscale|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[30.616, 32.018, 33.491, 35.263, 37.9...|
|000004.jpg|tulipes|[93.374, 96.787, 99.684, 102.755, 104...|
|000004.jpg|    lys|[207.335, 205.221, 205.107, 205.107, ...|
|000002.jpg|    lys|[135.315, 136.201, 135.087, 136.087, ...|
|000001.jpg|    lys|[22.399, 18.274, 60.137, 102.701, 108...|
+----------+-------+----------------------------------------+
only showing top 5 rows
Taille pixels_grayscale (attendu 4096) : 4096


## ML couleurs - prétraitement en bytes

In [20]:
# Prétraitement couleur (bytes) : RGB brut 0-255, encodé en BinaryType
from pyspark.sql.types import BinaryType

def pixels_to_bytes(pixels):
    if pixels is None:
        return None
    int_pixels = [int(p) for p in pixels]
    return struct.pack(f"{len(int_pixels)}B", *int_pixels)

bytes_udf = F.udf(pixels_to_bytes, BinaryType())

def preprocess_color_bytes(df):
    return df.withColumn("pixels_color_bytes", bytes_udf(F.col("pixels")))

train_color_bytes_df = preprocess_color_bytes(train_parsed_df)
test_color_bytes_df  = preprocess_color_bytes(test_parsed_df)

print("Aperçu après prétraitement :")
train_color_bytes_df.select("image_id", "label", "pixels_color_bytes").show(5, truncate=40)

expected_len = TARGET_W * TARGET_H * 3
check_len = (
    train_color_bytes_df
    .select(F.length(F.col("pixels_color_bytes")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_color_bytes en octets (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                      pixels_color_bytes|
+----------+-------+----------------------------------------+
|000005.jpg|    lys|[1A 24 0F 1A 26 11 1B 28 11 1D 2A 11 ...|
|000004.jpg|tulipes|[65 5F 41 69 62 45 6E 64 47 72 67 48 ...|
|000004.jpg|    lys|[C8 D1 DA C6 CF D7 C6 CF D6 C6 CF D6 ...|
|000002.jpg|    lys|[8E 83 8C 8F 84 8C 8E 83 8A 8F 84 8B ...|
|000001.jpg|    lys|[14 1B 05 0E 18 00 3A 3F 33 66 67 67 ...|
+----------+-------+----------------------------------------+
only showing top 5 rows
Taille pixels_color_bytes en octets (attendu 12288) : 12288


ValueError: X has 4096 features, but RandomForestClassifier is expecting 12288 features as input.

## ML couleurs normalisées


In [13]:
spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.python.worker.memory", "2g") \
    .getOrCreate()

In [14]:
FEATURE_COL = "pixels_norm"
LABEL_COL   = "label"

def to_numpy(df, feature_col=FEATURE_COL, label_col=LABEL_COL):
    # Ramène (image_id, label, features) du cluster vers le driver
    # Seul point de collect() du pipeline ML — exception documentée
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

# Collecte train / test (driver)
train_ids, X_train, y_train_raw = to_numpy(train_rgb_normalized_df)
test_ids,  X_test,  y_test_raw  = to_numpy(test_rgb_normalized_df)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")

# --- Encodage des labels (lys/tulipes -> 0/1) ---
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

# --- Entraînement RandomForest ---
rf_rgb_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_rgb_model.fit(X_train, y_train)

# --- Évaluation ---
y_pred = rf_rgb_model.predict(X_test)
y_pred_proba = rf_rgb_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy (couleurs normalisées) : {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

X_train shape : (10, 12288)
X_test  shape : (10, 12288)
Classes : {0: 'lys', 1: 'tulipes'}

Accuracy (couleurs normalisées) : 0.4000
              precision    recall  f1-score   support

         lys       0.43      0.60      0.50         5
     tulipes       0.33      0.20      0.25         5

    accuracy                           0.40        10
   macro avg       0.38      0.40      0.38        10
weighted avg       0.38      0.40      0.38        10



In [15]:
train_rgb_normalized_df.count()

10

## ML grayscale

In [18]:
## 4.1 - ML : Grayscale non normalisé (0-255)

LABEL_COL   = "label"
FEATURE_COL = "pixels_grayscale"

def to_numpy(df, feature_col, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"]    for r in rows]
    X         = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y         = [r[label_col]     for r in rows]
    return image_ids, X, y

train_ids, X_train, y_train_raw = to_numpy(train_preprocessed_gray_df, FEATURE_COL)
test_ids,  X_test,  y_test_raw  = to_numpy(test_preprocessed_gray_df,  FEATURE_COL)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

rf_gray_raw = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_gray_raw.fit(X_train, y_train)

y_pred       = rf_gray_raw.predict(X_test)
y_pred_proba = rf_gray_raw.predict_proba(X_test)

print(f"\nAccuracy (grayscale non normalisé) : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

predicted_labels = label_encoder.inverse_transform(y_pred)
confidences      = y_pred_proba.max(axis=1).tolist()

predictions_gray_raw_df = spark.createDataFrame(
    list(zip(test_ids, y_test_raw, predicted_labels.tolist(), confidences)),
    schema=StructType([
        StructField("image_id",        StringType(), False),
        StructField("true_label",      StringType(), False),
        StructField("predicted_label", StringType(), False),
        StructField("confidence",      FloatType(),  False),
    ]),
)

print("\nPrédictions (Spark DataFrame) :")
predictions_gray_raw_df.show(10, truncate=False)

X_train shape : (10, 4096)
X_test  shape : (10, 4096)
Classes : {0: 'lys', 1: 'tulipes'}

Accuracy (grayscale non normalisé) : 0.7000
              precision    recall  f1-score   support

         lys       0.75      0.60      0.67         5
     tulipes       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10


Prédictions (Spark DataFrame) :
+----------+----------+---------------+----------+
|image_id  |true_label|predicted_label|confidence|
+----------+----------+---------------+----------+
|000139.jpg|tulipes   |tulipes        |0.58      |
|000137.jpg|tulipes   |tulipes        |0.535     |
|000063.jpg|lys       |tulipes        |0.615     |
|000140.jpg|tulipes   |lys            |0.6       |
|000061.jpg|lys       |tulipes        |0.715     |
|000138.jpg|tulipes   |tulipes        |0.535     |
|000062.jpg|lys       |lys            |0.57    

## ML grayscale normalisé

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

LABEL_COL   = "label"
FEATURE_COL = "pixels_gray"

def to_numpy(df, feature_col=FEATURE_COL, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

train_ids, X_train_gray, y_train_raw = to_numpy(train_preprocessed_norm_gray_df)
test_ids,  X_test_gray,  y_test_raw  = to_numpy(test_preprocessed_norm_gray_df)

print(f"X_train_gray shape : {X_train_gray.shape}")
print(f"X_test_gray  shape : {X_test_gray.shape}")

label_encoder = LabelEncoder()
label_encoder.fit(y_train_raw)
y_train = label_encoder.transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

rf_gray_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_gray_model.fit(X_train_gray, y_train)

y_pred_gray       = rf_gray_model.predict(X_test_gray)
y_pred_proba_gray = rf_gray_model.predict_proba(X_test_gray)

acc_gray = accuracy_score(y_test, y_pred_gray)
print(f"\nAccuracy (grayscale normalisé) : {acc_gray:.4f}")
print(classification_report(y_test, y_pred_gray, target_names=label_encoder.classes_))

predicted_labels_gray = label_encoder.inverse_transform(y_pred_gray)
confidences_gray      = y_pred_proba_gray.max(axis=1).tolist()

predictions_gray_df = spark.createDataFrame(
    list(zip(test_ids, y_test_raw, predicted_labels_gray.tolist(), confidences_gray)),
    schema=StructType([
        StructField("image_id",        StringType(), False),
        StructField("true_label",      StringType(), False),
        StructField("predicted_label", StringType(), False),
        StructField("confidence",      FloatType(),  False),
    ]),
)

print("\nPrédictions grayscale (Spark DataFrame) :")
predictions_gray_df.show(10, truncate=False)

X_train_gray shape : (10, 4096)
X_test_gray  shape : (10, 4096)
Classes : {0: 'lys', 1: 'tulipes'}

Accuracy (grayscale normalisé) : 0.7000
              precision    recall  f1-score   support

         lys       0.75      0.60      0.67         5
     tulipes       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10


Prédictions grayscale (Spark DataFrame) :
+----------+----------+---------------+----------+
|image_id  |true_label|predicted_label|confidence|
+----------+----------+---------------+----------+
|000139.jpg|tulipes   |tulipes        |0.58      |
|000137.jpg|tulipes   |tulipes        |0.535     |
|000063.jpg|lys       |tulipes        |0.615     |
|000140.jpg|tulipes   |lys            |0.6       |
|000061.jpg|lys       |tulipes        |0.715     |
|000138.jpg|tulipes   |tulipes        |0.535     |
|000062.jpg|lys       |lys     